### Step0: Preparation

In [ ]:
install.packages('purrr')
pwd <- getwd()

### Step1: Show all databases on the Seqslab platform

In [ ]:
res <- DBI::dbGetQuery(sc, glue::glue("SHOW DATABASES")) %>%
  mutate(index = dplyr::row_number()) %>% select(index,namespace) 
options(width = 200)
print(res,row.names = FALSE)
db_dict <- setNames(as.list(res$namespace), res$index)

### Step2: Select the database and view  tables schema

In [ ]:
db_index <- readline(prompt = "Enter the index of the database: ")
db <- db_dict[[db_index]]
DBI::dbGetQuery(sc, glue::glue("USE {db}"))

#DBI::dbGetQuery(sc, glue::glue("SHOW TABLES IN {db}"))
tbs <- c(DBI::dbListTables(sc))
matched_items <- tbs[grepl("_delta_", tbs)]
results <- list()
for (i in seq_along(matched_items)) {
    tb <- matched_items[i]
    results[[tb]] <- DBI::dbGetQuery(sc, glue::glue("DESCRIBE {tb}"))
}
cat("\n---Database:", db, "---\n")
invisible(purrr::imap(results, function(tbl, name) {
  cat("\n---Table:", name, "---\n")
  print(tbl)
}))

### Step3: Query Data

#### Method 1 (Recommended for users familiar with R)

In [ ]:
# Recommended for users familiar with the tidyverse pipe in R
# Retrieve data using a tidyverse-style workflow.

# 1️⃣ Load the table from Spark
# Replace the table name "metagenomicgenefamilies_demo_delta_1115081894" with your own Spark table name.
table <- dplyr::tbl(sc,  "metagenomicgenefamilies_demo_delta_1115081894") |> 

# 2️⃣ Filter rows
  # You can change the condition inside filter() based on your analysis needs.
  # Example:
  #   dplyr::filter(species == "Escherichia coli")
  # To combine multiple conditions, use & (and) or | (or), for example:
  #   dplyr::filter(species == "Bacteroides vulgatus", abundanceRPKs > 10)
dplyr::filter(species=="Bacteroides vulgatus")|> 
  # 3️⃣ Select specific columns
  # Modify the column names inside select() to include only the columns you need.
  # Example:
  #   dplyr::select("geneFamilyName", "abundanceRPKs", "sampleId")
dplyr::select("geneFamilyName","abundanceRPKs","species")

# 4️⃣ Display the generated SQL query
# This shows the actual SQL statement sent to Spark, which is useful for debugging or learning.
dplyr::show_query(table)

# 5️⃣ Display the filtered table (will execute the query on Spark)
table

#### Method 2 (Recommended for users familiar with SQL)

In [ ]:
# Recommended for users familiar with SQL
# You can modify the SQL statement below to fit your requirements.
query="SELECT geneFamilyName, abundanceRPKs, species FROM metagenomicgenefamilies_demo_delta_1115081894 WHERE species = 'Bacteroides vulgatus' "
table <- DBI::dbGetQuery(sc, glue::glue("{query}"))
table
# The class of this table is data.frame, a local object where the data exists in local R memory.

### Step4: Convert Spark table to local R DataFrame

In [ ]:
# To collect the dataset into local memory for further plotting or statistical analysis.
local_table <- collect(table)
class(local_table)

### Demo. Compute summary statistics

In [ ]:
# R data.frame function
summary(local_table)

# Other sparklyr function
# https://spark.posit.co/packages/sparklyr/latest/reference/
sdf_describe(table)

### Demo. Saving plots to a file with `pdf()`

In [ ]:
# Step 1: Call the pdf command to start the plot
filename <- "test.pdf"
filepath <- paste(pwd,filename,sep='/')
pdf(file = filepath,   # The directory you want to save the file in
    width = 4, # The width of the plot in inches
    height = 4) # The height of the plot in inches

# Step 2: Create the plot with R code
plot(x = 1:10, 
     y = 1:10)
abline(v = 0) # Additional low-level plotting commands
text(x = 0, y = 1, labels = "Random text")

# Step 3: Run dev.off() to create the file!
dev.off()
print(paste("Your file was saved in",filepath))